## Completion finetuning using unsloth

This notebook makes use of unsloth to finetune a model for a completion task.
In this example we will finetune the llama 3.2 base model to generate ascii art. I would recommend using the unsloth library compared to just using the huggingface library as it requires less memory and is faster.

Adapted from unsloth notebooks, if something is broken check on:
https://unsloth.ai/

In [1]:
%%capture
!pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3  peft trl triton
!pip install --no-deps cut_cross_entropy unsloth_zoo
!pip install sentencepiece protobuf datasets huggingface_hub hf_transfer
!pip install --no-deps unsloth

### Load base model

In [2]:
from unsloth import FastLanguageModel
import torch
from google.colab import userdata


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="meta-llama/Llama-3.2-3B",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = False, # we will be using only LORA not QLORA
    token=userdata.get('HF_ACCESS_TOKEN')
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


    PyTorch 2.6.0+cu124 with CUDA 1204 (you have 2.8.0+cu126)
    Python  3.12.9 (you have 3.12.12)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=1 for more details


Switching to PyTorch attention since your Xformers is broken.

Unsloth: Xformers was not installed correctly.
Please install xformers separately first.
Then confirm if it's correctly installed by running:
python -m xformers.info

Longer error message:
xFormers can't load C++/CUDA extensions. xFormers was built for:
    PyTorch 2.6.0+cu124 with CUDA 1204 (you have 2.8.0+cu126)
    Python  3.12.9 (you have 3.12.12)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.10.9: Fast Llama patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

In [3]:
# This setting tells the tokenizer whether to automatically fix spacing and
# punctuation artifacts that appear when decoding tokens back into text.
# we had to use it because without it the ASCII art was not properly generated.
tokenizer.clean_up_tokenization_spaces = False

### Add lora to base model and patch with Unsloth

In [4]:
# More info about parameters: https://huggingface.co/docs/peft/v0.11.0/en/package_reference/lora#peft.LoraConfig
target_modules =  ["q_proj", "k_proj", "v_proj", "o_proj",
                   "gate_proj", "up_proj", "down_proj"]

# When adding special tokens
train_embeddings = False

if train_embeddings:
  target_modules = target_modules + ["lm_head"]

model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # rank of lora matrices according to paper not much loss when set relatively low
    target_modules = target_modules,  # On which modules of the llm the lora weights are used
    lora_alpha = 16, # scales the weights of the adapters (more influence on base model), 16 was recommended on reddit, usually 2*r
    lora_dropout = 0, # Default on 0.05 in tutorial but unsloth says 0 is better
    bias = "none",    # "none" is optimized
    use_gradient_checkpointing = "unsloth", #"unsloth" for very long context, decreases vram
    random_state = 3407,
    use_rslora = False,  # (Rank-Stabilized LoRA) normalizes to make different ranks behave consistently; can help at very low ranks (e.g., 4–8).
    loftq_config = None, # Use it only when you’re doing QLoRA (load_in_4bit=True) and see quality drop; if your base is 16-bit (plain LoRA), don’t use LoftQ.
)

Unsloth 2025.10.9 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [5]:
empty_prompt = """
{ascii_art}
"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func_no_prompt(examples):
  ascii_art_samples = examples["ascii"]
  training_prompts = []
  for ascii_art in ascii_art_samples:
      training_prompt = empty_prompt.format(ascii_art=ascii_art) + EOS_TOKEN
      training_prompts.append(training_prompt)
  return { "text" : training_prompts, }


from datasets import load_dataset
dataset = load_dataset("pookie3000/ascii-cats", split = "train")
dataset = dataset.map(formatting_prompts_func_no_prompt, batched = True)

README.md:   0%|          | 0.00/305 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/13.8k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/201 [00:00<?, ? examples/s]

Map:   0%|          | 0/201 [00:00<?, ? examples/s]

 ### Visualize dataset

In [6]:
for i, sample in enumerate(dataset):
    print(f"\nSample {i + 1}")
    print(sample["text"])
    if i > 2:
      break


Sample 1

    /\_/\           ___
   = o_o =_______    \ \ 
    __^      __(  \.__) )
(@)<_____>__(_____)____/
<|end_of_text|>

Sample 2

|\---/|
| o_o |
 \_^_/
<|end_of_text|>

Sample 3

 |\__/,|   (`\
 |_ _  |.--.) )
 ( T   )     /
(((^_(((/(((_/
<|end_of_text|>

Sample 4

   |\---/|
   | ,_, |
    \_`_/-..----.
 ___/ `   ' ,""+ \  
(__...'   __\    |`.___.';
  (_,...'(_,.`__)/'.....+
<|end_of_text|>


In [7]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    dataset_num_proc = 2, # CPU cores/CPU worker processes to preprocess/tokenize the dataset in parallel
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4, # process 4 batches before updating parameters (parameter update == step)
        num_train_epochs = 15,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none"
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/201 [00:00<?, ? examples/s]

In [8]:
trainer_stats = trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 201 | Num Epochs = 15 | Total steps = 390
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Step,Training Loss
1,3.760300
2,3.533000
3,4.388500
4,3.926100
5,3.530500
6,4.094000
7,3.802000
8,3.708200
9,3.464700
10,3.486200


### inference

In [9]:
from transformers import TextStreamer

def generate_ascii_art(model):
    FastLanguageModel.for_inference(model)
    inputs = tokenizer("", return_tensors="pt").to("cuda")
    text_streamer = TextStreamer(tokenizer, skip_special_tokens=True, skip_prompt=True)

    _ = model.generate(**inputs, streamer=text_streamer, max_new_tokens=500)  # no loop, no print

In [10]:
for _ in range(11):
  generate_ascii_art(model)


   |\      _,,,---,,_
   /,`.-'`'    -.  ;-;;,_
  |,4-  ) )-,_..;\ (  `'-'
 '---''(_/--'  `-'\_)  


  /\ ___ /\
 (  >   <  )
  \  >#<  /
  /       \
 /         \
 |         |
  \       /
  /// /// --


  /\_/\  (
 ( ^.^ ) _)
   \"/  (
 ( | | )
(__d b__)


   |,\__/|
   |  o o|  ,,
   (   T ) //
  .^`--^'^.
 `.  ;  .'
 | | | | |
((_((|))_))


 _
//
||              |\_/|
 \\  .-""""-._,' O O
  \\/         \  =_Y/=
   \    \       /`"`
    \   | /    |
    /  / -\   /
    `\ \\  | ||
      \_)) |_))



  |\__/,|
  |o o  |
  ( T   )
 .^`^--'^.
 `.  ;  .'  
 | | | | |
((_((|))_))


   |\---/|
   | >_< |
    \_`_/-..----.
 ___/ `   ' ,""+ \  
(__...'   __\    |`.___.';
  (_,...'(_,.`__)/'.....+


  |\__/|   (`\  
  |o  o| __ _) )  
  |  ~ |    /  
  (----)  //  
   || ||  ||  



       /)
       ((
        ))
   ,   //,
  /,\="=/,\
 /` o   o `\
=\:.  ^  .:/=
 /'***o***'\
( (         ) )
(,,)'-=-'(,,)



  /\_/\  (
 ( ^.^ ) _)
   \"/  (
 ( | | )
(__d b__)


.       .         
\-"'"-'/
 } 0

## Saving

### Save lora adapter

* Saves only the LoRA delta weights (not the base model).

* No merging happens.

* No GGUF file is created — it just uploads the adapter in Hugging Face format (e.g., .safetensors, adapter_config.json).

In [11]:
from peft import PeftModel
import os, glob

# Step 1: Prepare a local directory to store model + tokenizer files
hf_dir = "/tmp/hf_model"
os.makedirs(hf_dir, exist_ok=True)

# Step 2: Save tokenizer (creates tokenizer.json, special_tokens_map.json, etc.)
tokenizer.save_pretrained(hf_dir)

# Step 3: Save base model config
# If it's a PEFT (LoRA) model, extract the base model config properly
base_cfg = (model.base_model.config if isinstance(model, PeftModel) else model.config)
base_cfg.to_json_file(os.path.join(hf_dir, "config.json"))

# Step 4: Save LoRA adapter weights
model.save_pretrained(hf_dir)

# Step 5: Verification (ensure files exist before pushing)
print("config.json:", os.path.exists(f"{hf_dir}/config.json"))
print("tokenizer.json:", os.path.exists(f"{hf_dir}/tokenizer.json"))
print("some weights present:", bool(glob.glob(f"{hf_dir}/*.safetensors")))

# Step 6: Push the LoRA adapter (delta weights) to Hugging Face Hub
model.push_to_hub(
    "Hasnat5/Llama-3.2-3B-ascii-cats-lora",
    tokenizer=tokenizer,
    token=userdata.get('HF_ACCESS_TOKEN')
)


README.md:   0%|          | 0.00/560 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  60%|######    | 58.7MB / 97.3MB            

Saved model to https://huggingface.co/Hasnat5/Llama-3.2-3B-ascii-cats-lora


### Merge model with lora weights and save to gguf

You can then do inference locally with Ollama or llama.cpp

##### Popular quantization methods

- **q4_k_m**  
  4bit quantization. Low memory. All models you pull with ollama uses this quantization.
- **q8_0**  
  8bit quantization. Medium memory.
- **f16**  
  16 bit quantization. A lot of models are already in 16 bit so then no quantization happens
- **not_quantized**  
  Often same as f16.

In [12]:
# * Automatically merges the base model + LoRA adapter before exporting.
# * Then quantizes the merged model (to 4-bit Q4_K_M).
# * Outputs a single .gguf file containing the entire model (ready for inference in llama.cpp / Ollama).

from peft import PeftModel
import os, glob

# Step 1: Prepare the save directory
hf_dir = "/tmp/hf_model"
os.makedirs(hf_dir, exist_ok=True)

# Step 2: Save tokenizer (writes tokenizer.json, special_tokens_map.json, etc.)
tokenizer.save_pretrained(hf_dir)

# Step 3: Save the *base model* config as config.json
base_cfg = (model.base_model.config if isinstance(model, PeftModel) else model.config)
base_cfg.to_json_file(os.path.join(hf_dir, "config.json"))

# Step 4: Save LoRA adapter weights into the same folder
model.save_pretrained(hf_dir)

# Step 5: Verification (optional but good practice)
print("config.json:", os.path.exists(f"{hf_dir}/config.json"))
print("tokenizer.json:", os.path.exists(f"{hf_dir}/tokenizer.json"))
print("some weights present:", bool(glob.glob(f"{hf_dir}/*.safetensors")))

# Step 6: Push merged + quantized model as GGUF
model.push_to_hub_gguf(
    "Hasnat5/Llama-3.2-3B-ascii-cats-lora-q4_k_m-GGUF",
    tokenizer=tokenizer,
    quantization_method="q4_k_m",
    token=userdata.get('HF_ACCESS_TOKEN')
)


Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 2 files from cache to `/tmp/unsloth_gguf_q0v22u92`: 100%|██████████| 2/2 [01:39<00:00, 49.73s/it]


Successfully copied all 2 files from cache to `/tmp/unsloth_gguf_q0v22u92`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [02:35<00:00, 77.52s/it]


Unsloth: Merge process complete. Saved to `/tmp/unsloth_gguf_q0v22u92`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: All required system packages already installed!
Unsloth: Install llama.cpp and building - please wait 1 to 3 minutes
Unsloth: Cloning llama.cpp repository
Unsloth: Install GGUF and other packages
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['Llama-3.2-3B.F16.gguf']
Unsloth: 

...

  Llama-3.2-3B.Q4_K_M.gguf    :   2%|1         | 33.5MB / 2.02GB            

Uploading config.json...


No files have been modified since last commit. Skipping to prevent empty commit.


Unsloth: Successfully uploaded GGUF to https://huggingface.co/Hasnat5/Llama-3.2-3B-ascii-cats-lora-q4_k_m-GGUF
Unsloth: Cleaning up temporary files...


'Hasnat5/Llama-3.2-3B-ascii-cats-lora-q4_k_m-GGUF'

In [15]:
# Creates a GGUF representation of only the LoRA adapter (no merging).
# This exports just the LoRA delta weights in GGUF format.
# You must load it later alongside a compatible base GGUF during inference.

from peft import PeftModel
import os, glob

# Step 1: Prepare local directory
hf_dir = "/tmp/hf_model"
os.makedirs(hf_dir, exist_ok=True)

# Step 2: Save tokenizer (writes tokenizer.json, special_tokens_map.json, etc.)
tokenizer.save_pretrained(hf_dir)

# Step 3: Save base model config (config.json)
base_cfg = (model.base_model.config if isinstance(model, PeftModel) else model.config)
base_cfg.to_json_file(os.path.join(hf_dir, "config.json"))

# Step 4: Save the LoRA adapter weights into the same folder
model.save_pretrained(hf_dir)

# Step 5: Verify saved files
print("config.json:", os.path.exists(f"{hf_dir}/config.json"))
print("tokenizer.json:", os.path.exists(f"{hf_dir}/tokenizer.json"))
print("some weights present:", bool(glob.glob(f"{hf_dir}/*.safetensors")))

# Step 6: Push the LoRA-only GGUF to Hugging Face (not merged)
model.push_to_hub_gguf(
    "Hasnat5/Llama-3.2-3B-ascii-cats-lora-F16-GGUF",
    tokenizer=tokenizer,            # explicit named parameter
    quantization_method="f16",      # saves in float16 precision
    merge_lora=False,               # keeps adapter separate
    token=userdata.get('HF_ACCESS_TOKEN')
)